In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
BASE_DIR = Path.cwd().parents[0]
df = pd.read_csv(BASE_DIR/'diabetes.csv')
print(df.isnull().sum())
df.info()
print(df.head())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: 

## Checking missing values

In [3]:
for col in df.columns:
    if col != 'Outcome' and col != 'Pregnancies':
        mask = df[col] == 0
        print(df.loc[mask, col].value_counts())

Glucose
0    5
Name: count, dtype: int64
BloodPressure
0    35
Name: count, dtype: int64
SkinThickness
0    227
Name: count, dtype: int64
Insulin
0    374
Name: count, dtype: int64
BMI
0.0    11
Name: count, dtype: int64
Series([], Name: count, dtype: int64)
Series([], Name: count, dtype: int64)


# Handling missing values
### MEDIAN

In [4]:
missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness',
                'Insulin', 'BMI']
for col in missing_vals:
    notzero_mask = df[col] != 0
    zero_mask = df[col] == 0
    
    col_med = df.loc[notzero_mask, col].median()
    df.loc[zero_mask, col] = col_med
    print(f'Missing values in {col} replaced with median: {col_med}')

Missing values in Glucose replaced with median: 117.0
Missing values in BloodPressure replaced with median: 72.0
Missing values in SkinThickness replaced with median: 29.0
Missing values in Insulin replaced with median: 125.0
Missing values in BMI replaced with median: 32.3


# Splitting

### installing scikit-learn and imbalanced-learn so that they have compatible versions

In [5]:
import sys
!{sys.executable} -m pip install -i https://mirror-pypi.runflare.com/simple scikit-learn imbalanced-learn

Looking in indexes: https://mirror-pypi.runflare.com/simple


In [6]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

shuffdf = df.sample(frac=1, random_state=7)
X = shuffdf.drop(columns=['Outcome'])
y = shuffdf['Outcome']
                 
rawx_tr, rawx_ts, y_tr, y_ts = train_test_split(X, y, test_size=0.2, stratify=y, random_state=7)

# Applying SMOTE resampling because of imbalance data
smote = SMOTE(random_state=7)
rawx_tr_resampled, y_tr_resampled = smote.fit_resample(rawx_tr, y_tr)
# rename y
y_tr = y_tr_resampled

# Scaling

In [7]:
from sklearn.preprocessing import StandardScaler

my_scaler = StandardScaler()

x_tr = my_scaler.fit_transform(rawx_tr_resampled)
x_ts = my_scaler.transform(rawx_ts)

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier 

from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

# Logistic Regression

In [9]:
C_vals = [0.001, 0.01, 0.1]
for c in C_vals:
    log_model = LogisticRegression(
        max_iter=500,
        penalty='l2',
        C= c,
        random_state=7
    )
    
    log_model.fit(x_tr, y_tr)
    preds = log_model.predict(x_ts)

    print(f'C = {c}, Ac. = {accuracy_score(y_ts, preds):.5f}, Pr. = {precision_score(y_ts, preds):.5f}, Rec. = {recall_score(y_ts, preds):.5f}, f1 = {f1_score(y_ts, preds):.5f}')

C = 0.001, Ac. = 0.74675, Pr. = 0.62295, Rec. = 0.70370, f1 = 0.66087
C = 0.01, Ac. = 0.75974, Pr. = 0.63934, Rec. = 0.72222, f1 = 0.67826
C = 0.1, Ac. = 0.74026, Pr. = 0.61667, Rec. = 0.68519, f1 = 0.64912


C:\Users\ACER\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\ACER\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\ACER\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set

# KNN

In [10]:
k_vals = range(1, 16, 2)
for k in k_vals:
    knn_model = KNeighborsClassifier(
        n_neighbors=k, 
        weights='distance',
        metric='minkowski',
        p=2
    )

    knn_model.fit(x_tr, y_tr)
    preds = knn_model.predict(x_ts)

    print(f'K = {k}, Ac. = {accuracy_score(y_ts, preds):.5f}, Pr. = {precision_score(y_ts, preds):.5f}, Rec. = {recall_score(y_ts, preds):.5f}, f1 = {f1_score(y_ts, preds):.5f}')

K = 1, Ac. = 0.65584, Pr. = 0.50820, Rec. = 0.57407, f1 = 0.53913
K = 3, Ac. = 0.70779, Pr. = 0.56338, Rec. = 0.74074, f1 = 0.64000
K = 5, Ac. = 0.73377, Pr. = 0.59701, Rec. = 0.74074, f1 = 0.66116
K = 7, Ac. = 0.74026, Pr. = 0.60000, Rec. = 0.77778, f1 = 0.67742
K = 9, Ac. = 0.76623, Pr. = 0.62857, Rec. = 0.81481, f1 = 0.70968
K = 11, Ac. = 0.74026, Pr. = 0.58974, Rec. = 0.85185, f1 = 0.69697
K = 13, Ac. = 0.74675, Pr. = 0.60000, Rec. = 0.83333, f1 = 0.69767
K = 15, Ac. = 0.77273, Pr. = 0.62667, Rec. = 0.87037, f1 = 0.72868


# SVM

In [11]:
kernel = ['rbf', 'linear', 'sigmoid']
C_vals = [0.01, 0.1, 1, 10]
for ker in kernel:
    print(ker)
    for c in C_vals:
        svm_model = SVC(
            kernel=ker,
            C=c,
            random_state=7
        )

        svm_model.fit(x_tr, y_tr)
        preds = svm_model.predict(x_ts)
        
        print(f'C = {c}, Ac. = {accuracy_score(y_ts, preds):.5f}, Pr. = {precision_score(y_ts, preds):.5f}, Rec. = {recall_score(y_ts, preds):.5f}, f1 = {f1_score(y_ts, preds):.5f}')

rbf
C = 0.01, Ac. = 0.68831, Pr. = 0.53333, Rec. = 0.88889, f1 = 0.66667
C = 0.1, Ac. = 0.75974, Pr. = 0.61333, Rec. = 0.85185, f1 = 0.71318
C = 1, Ac. = 0.79221, Pr. = 0.69643, Rec. = 0.72222, f1 = 0.70909
C = 10, Ac. = 0.74675, Pr. = 0.63158, Rec. = 0.66667, f1 = 0.64865
linear
C = 0.01, Ac. = 0.75325, Pr. = 0.62903, Rec. = 0.72222, f1 = 0.67241
C = 0.1, Ac. = 0.75974, Pr. = 0.64407, Rec. = 0.70370, f1 = 0.67257
C = 1, Ac. = 0.75974, Pr. = 0.64407, Rec. = 0.70370, f1 = 0.67257
C = 10, Ac. = 0.75974, Pr. = 0.64407, Rec. = 0.70370, f1 = 0.67257
sigmoid
C = 0.01, Ac. = 0.75974, Pr. = 0.63934, Rec. = 0.72222, f1 = 0.67826
C = 0.1, Ac. = 0.76623, Pr. = 0.65000, Rec. = 0.72222, f1 = 0.68421
C = 1, Ac. = 0.66234, Pr. = 0.51515, Rec. = 0.62963, f1 = 0.56667
C = 10, Ac. = 0.66883, Pr. = 0.52000, Rec. = 0.72222, f1 = 0.60465


# Gaussian Naive Bayes

In [12]:
gnb_model = GaussianNB(
    priors=None,
    var_smoothing=1e-10
)

gnb_model.fit(x_tr, y_tr)
gnb_model.predict(x_ts)

print(f'Ac. = {accuracy_score(y_ts, preds):.5f}, Pr. = {precision_score(y_ts, preds):.5f}, Rec. = {recall_score(y_ts, preds):.5f}, f1 = {f1_score(y_ts, preds):.5f}')

Ac. = 0.66883, Pr. = 0.52000, Rec. = 0.72222, f1 = 0.60465


# Random Forest

In [13]:
tree_nums = [50, 100, 200]
depth = [5, 10, 20]
for t in tree_nums:
    print(f'number of trees: {t}')
    for d in depth:
        rf_model = RandomForestClassifier(
            n_estimators=t,
            max_depth=d,
            min_samples_split=2,
            random_state=7
        )

        rf_model.fit(x_tr, y_tr)
        preds = rf_model.predict(x_ts)
    
        print(f'Depth = {d}, Ac. = {accuracy_score(y_ts, preds):.5f}, Pr. = {precision_score(y_ts, preds):.5f}, Rec. = {recall_score(y_ts, preds):.5f}, f1 = {f1_score(y_ts, preds):.5f}')

number of trees: 50
Depth = 5, Ac. = 0.78571, Pr. = 0.65217, Rec. = 0.83333, f1 = 0.73171
Depth = 10, Ac. = 0.78571, Pr. = 0.66154, Rec. = 0.79630, f1 = 0.72269
Depth = 20, Ac. = 0.77273, Pr. = 0.66102, Rec. = 0.72222, f1 = 0.69027
number of trees: 100
Depth = 5, Ac. = 0.78571, Pr. = 0.65672, Rec. = 0.81481, f1 = 0.72727
Depth = 10, Ac. = 0.79870, Pr. = 0.68254, Rec. = 0.79630, f1 = 0.73504
Depth = 20, Ac. = 0.78571, Pr. = 0.67797, Rec. = 0.74074, f1 = 0.70796
number of trees: 200
Depth = 5, Ac. = 0.78571, Pr. = 0.66154, Rec. = 0.79630, f1 = 0.72269
Depth = 10, Ac. = 0.81169, Pr. = 0.70492, Rec. = 0.79630, f1 = 0.74783
Depth = 20, Ac. = 0.79870, Pr. = 0.69492, Rec. = 0.75926, f1 = 0.72566


# Multi Layer Perceptron

In [14]:
act_func = ['relu', 'tanh']
alphas = [0.001, 0.01, 0.1]
hidden_size = [(8, ), (16,), (32, 16)]
for f in act_func:
    print(f'Activation: {f}')
    for s in hidden_size:
        print(f'HiddenLayerSize: {s}')
        for a in alphas:
            mlp_model = MLPClassifier(
                activation=f,
                hidden_layer_sizes=s,
                solver='adam',
                alpha=a,
                random_state=7
            )

            mlp_model.fit(x_tr, y_tr)
            preds = mlp_model.predict(x_ts)
            
            print(f'reg strength = {a}, Ac. = {accuracy_score(y_ts, preds):.5f}, Pr. = {precision_score(y_ts, preds):.5f}, Rec. = {recall_score(y_ts, preds):.5f}, f1 = {f1_score(y_ts, preds):.5f}')

Activation: relu
HiddenLayerSize: (8,)


C:\Users\ACER\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


reg strength = 0.001, Ac. = 0.77922, Pr. = 0.63889, Rec. = 0.85185, f1 = 0.73016


C:\Users\ACER\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


reg strength = 0.01, Ac. = 0.77922, Pr. = 0.63889, Rec. = 0.85185, f1 = 0.73016


C:\Users\ACER\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


... (26 outputs left out)

reg strength = 0.001, Ac. = 0.76623, Pr. = 0.63636, Rec. = 0.77778, f1 = 0.70000


C:\Users\ACER\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


reg strength = 0.01, Ac. = 0.76623, Pr. = 0.63636, Rec. = 0.77778, f1 = 0.70000
reg strength = 0.1, Ac. = 0.77922, Pr. = 0.65152, Rec. = 0.79630, f1 = 0.71667


C:\Users\ACER\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [15]:
#🏆 Top Performers 
#Model	    Best Configuration	    Accuracy	Precision	  Recall	 F1
#SVM           (rbf) C=1	         0.792	     0.696	      0.722	    0.678
#MLP (relu)	hidden=(8,), α=0.001	 0.779	     0.638	      0.851   	0.730
#Random Forest	n=200, depth=10	     0.811	     0.704	      0.796	    0.747
#KNN	         K=15	             0.772	     0.626	      0.870 	0.728
#Logistic	     C=0.01	             0.759	     0.639	      0.722	    0.678

we are building a Simulator, not just a Detector.

Detector: Use the most accurate model (SVM/RF). You just need a Yes/No answer.

Simulator: Use a transparent, mathematical model (Logistic Regression). You need to show the gradient of change.

Logistic Regression gives the coefficient mapping for free. It allows you to say: "If you change X, the risk changes by Y." With a Random Forest, you'd have to build a whole second system just to guess what the Random Forest is thinking. The Model is Just a Math Formula:
our trained model is literally just a list of 8 numbers (coefficients) and 1 starting number (intercept).
Risk Score = Intercept + (Coef_Glucose × Glucose) + (Coef_BMI × BMI) + ...

The Slider Update Calculation:
Original BMI: 35.0
New BMI: 33.0

BMI Coefficient: 0.12 (example)
Change in Risk Score: 0.12 * (33.0 - 35.0) = -0.24

Performance: This is 1 multiplication and 1 addition. Your computer can do this in 0.0000001 seconds. The slider feels fluid and real-time.

if we used SVM for example (acc=0.792), It must calculate the mathematical distance between the new data point and 200+ "support vectors" (memorized edge cases from the training data).

but we have to increase the metrics of the log reg we wanna use; so we manipulate the threshold so that more '1' for outcome is caught:

# tuned Logistic Regression

In [16]:
log_model = LogisticRegression(
    max_iter=500,
    penalty='l2',
    C=0.01,
    random_state=7
    )
log_model.fit(x_tr, y_tr)
    
# probabilities instead of class predictions
probs = log_model.predict_proba(x_ts)[:, 1]    # [:, 1]  "All rows, column 1" → Probability of Class 1 (Diabetic)

thresh = [0.35, 0.4, 0.45]
for t in thresh:
    # Lower threshold to catch more diabetics
    threshold = t
    preds_tuned = (probs >= threshold).astype(int)
    
    print(f"Tuned Accuracy: {accuracy_score(y_ts, preds_tuned):.5f}")
    print(f"Tuned Precision: {precision_score(y_ts, preds_tuned):.5f}")
    print(f"Tuned Recall: {recall_score(y_ts, preds_tuned):.5f}")
    print(f"Tuned F1: {f1_score(y_ts, preds_tuned):.5f}")
    print()

C:\Users\ACER\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Tuned Accuracy: 0.68182
Tuned Precision: 0.52577
Tuned Recall: 0.94444
Tuned F1: 0.67550

Tuned Accuracy: 0.70130
Tuned Precision: 0.54545
Tuned Recall: 0.88889
Tuned F1: 0.67606

Tuned Accuracy: 0.74675
Tuned Precision: 0.59494
Tuned Recall: 0.87037
Tuned F1: 0.70677



# Final training: LogisticRegression: C=0.01, Threshold=0.45

In [17]:
final_model = LogisticRegression(
    max_iter=500,
    penalty='l2',
    C=0.01,
    random_state= 7
    )
final_model.fit(x_tr, y_tr)
    
# probabilities instead of class predictions
probs = final_model.predict_proba(x_ts)[:, 1]

threshold = 0.45
preds_tuned = (probs >= threshold).astype(int)
    
print(f"Tuned Accuracy: {accuracy_score(y_ts, preds_tuned):.5f}")
print(f"Tuned Precision: {precision_score(y_ts, preds_tuned):.5f}")
print(f"Tuned Recall: {recall_score(y_ts, preds_tuned):.5f}")
print(f"Tuned F1: {f1_score(y_ts, preds_tuned):.5f}")

C:\Users\ACER\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Tuned Accuracy: 0.74675
Tuned Precision: 0.59494
Tuned Recall: 0.87037
Tuned F1: 0.70677
